<a href="https://colab.research.google.com/github/Aleeza-Nadeem/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aleeza-Nadeem/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane / rule idea:** pages that rank in the top 20 but get a lower CTR than is typical for their position are leaving clicks on the table. The rule scores each page by the estimated clicks left on the table and the action is to rewrite the title and meta description.

**Inputs used:** only past-window, observed fields (impressions, CTR, position, staleness, trend). The ready-made flag columns, `health_score` and `is_initial_refresh_candidate` are NOT used, to avoid label-derived inputs. All numbers are observed / measured / directional, for decision support only.

**Data:** the public anonymized starter dataset (~30k pages).

In [1]:
import pandas as pd, numpy as np, json, os
url = "https://huggingface.co/datasets/FlyRank/internship-starter/resolve/main/content_refresh_anonymized.csv"
df = pd.read_csv(url)
print(df.shape)
df[["ctr","avg_position","impressions_90d","days_since_last_update","trend_pct"]].describe()

(30000, 53)


,ctr,avg_position,impressions_90d,days_since_last_update,trend_pct
count,30000.000000,30000.00000,30000.000000,30000.000000,26612.000000
mean,0.510733,16.34238,5200.366300,46.098300,-4.785969
std,3.279162,15.21679,16838.019547,42.078709,473.861780
min,0.000000,0.00000,1.000000,1.000000,-100.000000
25%,0.000000,6.20000,81.000000,20.000000,-62.600000
50%,0.070000,10.80000,731.000000,20.000000,-33.500000
75%,0.290000,22.30000,3615.250000,104.000000,0.000000
max,100.000000,245.00000,517715.000000,373.000000,44900.000000


## 1) Signal checks (bucket table + n each)

### Signal A — staleness (the signal behind FlyRank's refresh flags)
*Claim to test: pages not updated for longer show worse recent trend.*

In [2]:
a = df.dropna(subset=["days_since_last_update","trend_pct"]).copy()
a["bucket"] = pd.qcut(a["days_since_last_update"], 4, duplicates="drop")
tA = a.groupby("bucket", observed=True).agg(
    n=("content_id","size"),
    median_trend_pct=("trend_pct","median"),
    share_declining=("trend_pct", lambda s: (s<0).mean()))
print(tA)
rho_A = a["days_since_last_update"].corr(a["trend_pct"], method="spearman")
print("n total:", len(a), "| Spearman(staleness, trend):", round(rho_A,3))
trend_dir = tA["median_trend_pct"].diff().dropna()
if rho_A < -0.05 and (trend_dir < 0).all():   verdict_A = "CONFIRMED"
elif rho_A > 0.05 and (trend_dir > 0).all():  verdict_A = "OPPOSITE"
elif abs(rho_A) <= 0.05:                      verdict_A = "FALSE"
else:                                         verdict_A = "MIXED"
print("Suggested verdict A:", verdict_A)

                    n  median_trend_pct  share_declining
bucket                                                  
(0.999, 20.0]   12992             -41.4         0.758698
(20.0, 22.0]     3493             -10.7         0.593186
(22.0, 104.0]    9878             -33.6         0.769387
(104.0, 373.0]    249             -54.4         0.746988
n total: 26612 | Spearman(staleness, trend): 0.072
Suggested verdict A: MIXED


**Verdict A: MIXED**

**Why:** Staleness does not line up cleanly with decline. The stalest bucket (104 to 373 days, n=249) has the worst median trend (-54.4%), but the freshest bucket (1 to 20 days, n=12,992) is also weak (-41.4%), the 20 to 22 day bucket is the mildest (-10.7%), and the Spearman correlation is slightly positive (+0.07), the opposite of the expectation. About three quarters of pages are declining in every bucket except one, so staleness alone does not separate them. Observed and directional only; the rule does not lean on staleness.

### Signal B — CTR vs position (the signal behind the CTR-fix logic) — flag-linked
*Claim to test: CTR falls as position gets worse. The rule leans on this.*

In [3]:
b = df[(df.impressions_90d >= 100) & df.avg_position.notna() & df.ctr.notna()].copy()
b["pos_bucket"] = pd.cut(b["avg_position"], [0,3,5,10,20,100])
tB = b.groupby("pos_bucket", observed=True).agg(
    n=("content_id","size"), median_ctr=("ctr","median"))
print(tB)
mono = tB["median_ctr"].is_monotonic_decreasing
print("n total:", len(b), "| CTR falls as position worsens:", mono)
verdict_B = "CONFIRMED" if mono else "MIXED"
print("Suggested verdict B:", verdict_B)

               n  median_ctr
pos_bucket                  
(0, 3]       555        0.19
(3, 5]      1964        0.32
(5, 10]     6696        0.21
(10, 20]    5876        0.15
(20, 100]   6915        0.04
n total: 22006 | CTR falls as position worsens: False
Suggested verdict B: MIXED


**Verdict B: MIXED**

**Why:** Median CTR falls as position worsens from the 3 to 5 bucket (0.32, n=1,964) down to 20 to 100 (0.04, n=6,915), but the top-3 bucket (0.19, n=555) sits below the 3 to 5 bucket, so the pattern is not strictly monotonic. The assumption holds for most of the range, so the rule is usable, but the top-3 benchmark is noisy and I treat top-3 picks with care. Also, `ctr` has invalid values above 1, which the rule excludes.

## 2) The rule — score, ONE reason code, action label

**Score** = impressions_90d × (typical CTR for the page's position bucket − the page's own CTR), floored at 0. This is estimated clicks left on the table, a directional measure.
**Reason code:** `LOW_CTR_FOR_POSITION` · **Action:** Rewrite title and meta description.

In [4]:
r = b[(b.avg_position <= 20) & (b.ctr <= 1)].copy()   # drop invalid CTR > 1
r["expected_ctr"] = r.groupby("pos_bucket", observed=True)["ctr"].transform("median")
r["ctr_gap"] = (r["expected_ctr"] - r["ctr"]).clip(lower=0)
r["score"] = r["impressions_90d"] * r["ctr_gap"]   # estimated clicks left on the table (CTR is a fraction)
r["reason_code"] = np.where(r["score"] > 0, "LOW_CTR_FOR_POSITION", "NONE")
r["action"] = np.where(r["score"] > 0, "Rewrite title and meta description", "No action")
queue = r[r.score > 0].sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

os.makedirs("work/outputs", exist_ok=True)
queue[["rank","content_id","score","reason_code","action"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False)
json.dump({"n_scored": int(len(r)), "n_queue": int(len(queue)),
           "verdict_A_staleness": verdict_A, "verdict_B_ctr_vs_position": verdict_B,
           "rule": "impressions_90d x (median CTR for position bucket - own CTR)"},
          open("work/outputs/baseline_action_score_metrics.json","w"), indent=2)
print("Scored:", len(r), "| In queue:", len(queue))
queue.head(10)[["rank","content_id","score","avg_position","ctr","impressions_90d"]]

Scored: 14324 | In queue: 6966


,rank,content_id,score,avg_position,ctr,impressions_90d
0,1,content_5fe46e04994d,77657.25,4.2,0.14,517715
1,2,content_36ff89c8214e,41313.58,7.3,0.05,295097
2,3,content_73c54f78c06a,40652.97,4.7,0.10,213963
3,4,content_c8e9d6ab9013,39648.82,9.7,0.00,208678
4,5,content_8451fc6f034d,38100.16,2.3,0.03,272144
5,6,content_c84a0ab98e90,35723.36,7.8,0.03,223271
6,7,content_453722754fea,25214.22,7.6,0.01,140079
7,8,content_1a9e894be2e2,24970.80,4.0,0.23,416180
8,9,content_c1fe78bc4e37,21448.80,7.5,0.03,134055
9,10,content_42d423551e2c,21330.40,4.8,0.09,106652


## 3) Top-10 review — action, why it's there, what would make it wrong
*Read the draft lines below, then rewrite at least the ones that don't fit in your own words.*

In [5]:
top = queue.head(10).copy()
def why(x):
    return (f"Position {x.avg_position:.1f}; CTR {x.ctr:.3f} vs typical {x.expected_ctr:.3f} "
            f"for that position; {int(x.impressions_90d):,} impressions in 90 days.")
top["why"] = top.apply(why, axis=1)
for _, x in top.iterrows():
    print(f"#{x['rank']} | {x.action}\n   Why: {x.why}\n")

#1 | Rewrite title and meta description
   Why: Position 4.2; CTR 0.140 vs typical 0.290 for that position; 517,715 impressions in 90 days.

#2 | Rewrite title and meta description
   Why: Position 7.3; CTR 0.050 vs typical 0.190 for that position; 295,097 impressions in 90 days.

#3 | Rewrite title and meta description
   Why: Position 4.7; CTR 0.100 vs typical 0.290 for that position; 213,963 impressions in 90 days.

#4 | Rewrite title and meta description
   Why: Position 9.7; CTR 0.000 vs typical 0.190 for that position; 208,678 impressions in 90 days.

#5 | Rewrite title and meta description
   Why: Position 2.3; CTR 0.030 vs typical 0.170 for that position; 272,144 impressions in 90 days.

#6 | Rewrite title and meta description
   Why: Position 7.8; CTR 0.030 vs typical 0.190 for that position; 223,271 impressions in 90 days.

#7 | Rewrite title and meta description
   Why: Position 7.6; CTR 0.010 vs typical 0.190 for that position; 140,079 impressions in 90 days.

#8 | Rewrite 

### My review of the top 10 (action | why it is there | what would make it wrong)

All ten: **Rewrite title and meta description** (reason code `LOW_CTR_FOR_POSITION`). Numbers are observed, directional, decision-support only.

1. **#1** — Position 4.2, CTR 0.14 vs typical 0.32, 517,715 impressions. *Wrong if* the impressions come from a few broad queries where people rarely click whatever the title says.
2. **#2** — Position 7.3, CTR 0.05 vs typical 0.21, 295,097 impressions. *Wrong if* a search feature (snippet, AI answer) at this position takes the clicks, so a new title would not help.
3. **#3** — Position 4.7, CTR 0.10 vs typical 0.32, 213,963 impressions. *Wrong if* the page ranks for queries it does not really answer, so the problem is intent match, not the title.
4. **#4** — Position 9.7, CTR 0.000 vs typical 0.21, 208,678 impressions. *Wrong if* a CTR of zero on this many impressions is a tracking or data issue, not a weak title. Check the data first.
5. **#5** — Position 2.3, CTR 0.03 vs typical 0.19, 272,144 impressions. *Wrong if* the top-3 benchmark is off: the top-3 median CTR (0.19) is lower than the 3 to 5 median (0.32) in this data.
6. **#6** — Position 7.8, CTR 0.03 vs typical 0.21, 223,271 impressions. *Wrong if* the average position hides a recent drop, so low CTR reflects a lower rank rather than a poor title.
7. **#7** — Position 4.0, CTR 0.23 vs typical 0.32, 416,180 impressions. *Wrong if* the gap is just noise: this is the smallest gap in the top 10 and it ranks high only because of its very large impressions.
8. **#8** — Position 7.6, CTR 0.01 vs typical 0.21, 140,079 impressions. *Wrong if* the searches are mostly branded or navigational, or the CTR figure is a data error.
9. **#9** — Position 4.8, CTR 0.09 vs typical 0.32, 106,652 impressions. *Wrong if* ads or other features above the result take the clicks, so rewriting the title changes nothing.
10. **#10** — Position 7.5, CTR 0.03 vs typical 0.21, 134,055 impressions. *Wrong if* the average position hides a wide spread across queries, with some at page 2 and some at position 1.


## 4) Weak picks
*Which of the top 10 am I least sure about, and why?* Write 2–3 sentences under the table.

In [6]:
weak = top[(top.days_with_impressions < 30) | (top.impressions_90d < 200)]
weak[["rank","content_id","days_with_impressions","impressions_90d","avg_position"]]

,rank,content_id,days_with_impressions,impressions_90d,avg_position


**My weak picks:** The weak-picks table above is empty, so none of the top 10 has fewer than 30 impression days or fewer than 200 impressions. The risk is different: all ten are very high-impression pages (roughly 100k to 500k impressions, against a dataset median of 731), so the queue is dominated by a few heavy-tail pages. #4 (CTR 0.000 on about 208k impressions) may be a data problem rather than a weak title, and #7 has the smallest gap, so it is the least certain pick. Treat the queue as decision-support to be checked manually, not as proof that a rewrite will lift clicks.

## 5) Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [7]:
chk = pd.read_csv("work/outputs/baseline_action_score.csv")
assert len(chk) > 0 and {"score","reason_code","action"} <= set(chk.columns)
assert chk["reason_code"].nunique() == 1          # ONE reason code
print("CSV OK:", chk.shape, "| signal verdicts:", verdict_A, verdict_B)

CSV OK: (6966, 5) | signal verdicts: MIXED MIXED
